# PyTorch Lightning fraud classifier

Run this in a separate fresh Colab runtime from Keras to avoid package conflicts. Record the actual Flash install attempt and the route used; do not claim an install result before performing the attempt.

## Setup and shared data

In [ ]:
from pathlib import Path
import os, subprocess
repo_root = Path.cwd()
if not (repo_root / "src" / "data.py").exists():
    repo_root = Path("/content/credit-card-fraud-detection")
    if not (repo_root / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git", str(repo_root)], check=True)
    os.chdir(repo_root)
print("Project root:", Path.cwd())


%pip install -q -r requirements/base.txt
%pip install -q -r requirements/lightning.txt
!python src/data.py
# In the full team repository, run the existing setup/data cells first.
# The team's src/data.py must expose load_splits() as agreed in the project plan.
from pathlib import Path
import sys, random, time, platform
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() / "src"))
from data import load_splits
from evaluate import evaluate_model, permutation_importance
SEED = 42
random.seed(SEED); np.random.seed(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torchmetrics.classification import BinaryAveragePrecision
L.seed_everything(42, workers=True)

(X_train, y_train, X_val, y_val, X_test, y_test,
 row_id_val, amount_val, row_id_test, amount_test) = load_splits()
feature_names = list(X_train.columns) if hasattr(X_train, "columns") else [f"x{i}" for i in range(X_train.shape[1])]
X_train = np.asarray(X_train, dtype=np.float32); X_val = np.asarray(X_val, dtype=np.float32); X_test = np.asarray(X_test, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int64).reshape(-1); y_val = np.asarray(y_val, dtype=np.int64).reshape(-1); y_test = np.asarray(y_test, dtype=np.int64).reshape(-1)
fraud_weight = float((y_train == 0).sum() / (y_train == 1).sum())


In [ ]:
class FraudNet(L.LightningModule):
    def __init__(self, n_features, pos_weight):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1),
        )
        self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float32))
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        self.val_ap = BinaryAveragePrecision()

    def forward(self, x): return self.net(x).squeeze(1)
    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y.float())
        self.log("train_loss", loss, on_epoch=True, prog_bar=False)
        return loss
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y.float())
        self.val_ap.update(torch.sigmoid(logits), y)
        self.log("val_loss", loss, on_step=False, on_epoch=True)
        self.log("val_pr_auc", self.val_ap, on_step=False, on_epoch=True, prog_bar=True)
    def configure_optimizers(self): return torch.optim.Adam(self.parameters(), lr=0.001)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)


## Train and evaluate

In [ ]:
model = FraudNet(X_train.shape[1], fraud_weight)
Path("models").mkdir(exist_ok=True)
checkpoint = ModelCheckpoint(dirpath="models/lightning_checkpoints", monitor="val_pr_auc", mode="max", save_top_k=1, filename="best-{epoch:02d}-{val_pr_auc:.4f}")
stop = EarlyStopping(monitor="val_pr_auc", mode="max", patience=5)
trainer = L.Trainer(max_epochs=30, accelerator="auto", devices=1, callbacks=[checkpoint, stop], log_every_n_steps=20, enable_progress_bar=True)
start = time.perf_counter()
trainer.fit(model, train_loader, val_loader)
train_seconds = time.perf_counter() - start
best_model = FraudNet.load_from_checkpoint(checkpoint.best_model_path, n_features=X_train.shape[1], pos_weight=fraud_weight)
best_model.eval(); device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); best_model.to(device)
def predict_probabilities(X):
    loader = DataLoader(torch.from_numpy(np.asarray(X, dtype=np.float32)), batch_size=8192, shuffle=False)
    values=[]
    with torch.no_grad():
        for xb in loader: values.append(torch.sigmoid(best_model(xb.to(device))).cpu().numpy())
    return np.concatenate(values)
p_val = predict_probabilities(X_val)
start = time.perf_counter()
p_test = predict_probabilities(X_test)
predict_seconds = time.perf_counter() - start
import shutil
shutil.copy2(checkpoint.best_model_path, "models/lightning.ckpt")
result = evaluate_model("lightning", y_val, p_val, y_test, p_test, row_id_val, row_id_test,
    train_seconds, predict_seconds,
    notes="Plain PyTorch Lightning used; Flash install outcome: record your actual result. Shared 64-32 network; class-weighted BCE; early stopping on validation PR-AUC.",
    hardware=f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
result


## Validation permutation importance

In [ ]:
def lightning_predict(frame):
    return predict_probabilities(np.asarray(frame, dtype=np.float32))
perm = permutation_importance(lightning_predict, pd.DataFrame(X_val, columns=feature_names), y_val, n_repeats=3, random_state=42)
Path("results").mkdir(exist_ok=True)
perm.to_csv("results/perm_lightning.csv", index=False)
perm.head(10)